# 🏈 College Football Bowl Game Predictor
## Exploratory Data Analysis (EDA)
**UE24CS352A — Machine Learning Mini-Project**  
**Member 1: Data + Preprocessing + EDA**

---
### Plots produced:
1. Home vs Away Win Distribution  
2. Average Points Distribution  
3. Average Rushing vs Passing Yards  
4. Correlation Heatmap of Features  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, sys

# ── Run preprocessing first if cleaned file doesn't exist ──
CLEAN_CSV = 'data/cfb_cleaned.csv'
RAW_CSV   = 'data/cfb_box-scores_2002-2025.csv'
RESULTS   = 'results/'

if not os.path.exists(CLEAN_CSV):
    print('Running preprocessing.py first...')
    sys.path.insert(0, 'src')
    import preprocessing
    preprocessing.main()

df = pd.read_csv(CLEAN_CSV, parse_dates=['date'])
print(f'Loaded cleaned data: {df.shape}')
df.head(3)

Running preprocessing.py first...


ModuleNotFoundError: No module named 'preprocessing'

---
## Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nDate range:', df['date'].min(), '→', df['date'].max())
print('\nTarget distribution:')
print(df['target'].value_counts().rename({1: 'Home Win', 0: 'Away Win'}))
print('\nMissing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## Plot 1 — Home vs Away Win Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Home vs Away Win Distribution', fontsize=16, fontweight='bold')

# ── Bar chart ──
counts = df['target'].value_counts().sort_index()
labels = ['Away Win', 'Home Win']
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(labels, counts.values, color=colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
axes[0].set_ylabel('Number of Games')
axes[0].set_title('Win Counts')
axes[0].set_ylim(0, counts.max() * 1.15)

# ── Pie chart ──
axes[1].pie(
    counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.05, 0.05),
    shadow=True
)
axes[1].set_title('Win Proportion')

plt.tight_layout()
plt.savefig(RESULTS + 'win_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/win_distribution.png')

---
## Plot 2 — Average Points Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Average Points Distribution', fontsize=16, fontweight='bold')

home_pts_col = 'home_avg_pts_scored'
away_pts_col = 'away_avg_pts_scored'

# ── Histogram overlay ──
axes[0].hist(df[home_pts_col].dropna(), bins=40, alpha=0.6,
             color='#2ecc71', label='Home Team Avg Pts')
axes[0].hist(df[away_pts_col].dropna(), bins=40, alpha=0.6,
             color='#e74c3c', label='Away Team Avg Pts')
axes[0].axvline(df[home_pts_col].mean(), color='#27ae60', linestyle='--',
                linewidth=2, label=f'Home Mean: {df[home_pts_col].mean():.1f}')
axes[0].axvline(df[away_pts_col].mean(), color='#c0392b', linestyle='--',
                linewidth=2, label=f'Away Mean: {df[away_pts_col].mean():.1f}')
axes[0].set_xlabel('Average Points Scored')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Points Distribution (Home vs Away)')
axes[0].legend()

# ── Boxplot: points by outcome ──
data_box = pd.DataFrame({
    'Avg Points': pd.concat([df[home_pts_col], df[away_pts_col]]),
    'Role'      : ['Home'] * len(df) + ['Away'] * len(df),
    'Outcome'   : pd.concat([
        df['target'].map({1: 'Won', 0: 'Lost'}),
        df['target'].map({0: 'Won', 1: 'Lost'})
    ])
})
sns.boxplot(data=data_box, x='Role', y='Avg Points', hue='Outcome',
            palette={'Won': '#2ecc71', 'Lost': '#e74c3c'}, ax=axes[1])
axes[1].set_title('Avg Points by Role & Outcome')
axes[1].set_ylabel('Average Points Scored')

plt.tight_layout()
plt.savefig(RESULTS + 'feature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/feature_distribution.png')

---
## Plot 3 — Average Rushing vs Passing Yards

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rushing vs Passing Yards — Home & Away Teams', fontsize=16, fontweight='bold')

# ── Scatter: rushing vs passing (home team) ──
scatter_colors = df['target'].map({1: '#2ecc71', 0: '#e74c3c'})
axes[0].scatter(df['home_avg_rush_yards'], df['home_avg_pass_yards'],
                c=scatter_colors, alpha=0.4, s=10)
axes[0].set_xlabel('Avg Rushing Yards')
axes[0].set_ylabel('Avg Passing Yards')
axes[0].set_title('Home Team: Rush vs Pass (Green=Win, Red=Loss)')
won_patch  = mpatches.Patch(color='#2ecc71', label='Home Won')
lost_patch = mpatches.Patch(color='#e74c3c', label='Home Lost')
axes[0].legend(handles=[won_patch, lost_patch])

# ── Bar: mean rush & pass yards for winners vs losers ──
metrics = ['home_avg_rush_yards', 'home_avg_pass_yards',
           'away_avg_rush_yards', 'away_avg_pass_yards']
won  = df[df['target'] == 1][metrics].mean()
lost = df[df['target'] == 0][metrics].mean()
x = np.arange(len(metrics))
width = 0.35
axes[1].bar(x - width/2, won.values, width, label='Home Won', color='#2ecc71', alpha=0.8)
axes[1].bar(x + width/2, lost.values, width, label='Home Lost', color='#e74c3c', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Home\nRush', 'Home\nPass', 'Away\nRush', 'Away\nPass'],
                        fontsize=9)
axes[1].set_ylabel('Yards')
axes[1].set_title('Mean Yards by Outcome')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS + 'rushing_passing_yards.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rushing_passing_yards.png')

---
## Plot 4 — Correlation Heatmap

In [ ]:
# Select only the engineered feature columns + target
feat_cols = [c for c in df.columns if c.startswith('home_avg_') or c.startswith('away_avg_')]
feat_cols += ['target']

corr = df[feat_cols].corr()

# Shorter labels for readability
short_labels = [c.replace('home_avg_', 'H_').replace('away_avg_', 'A_')
                  .replace('pts_scored', 'pts').replace('pts_allowed', 'pts_alw')
                  .replace('pass_yds', 'pass').replace('rush_yds', 'rush')
                  .replace('total_yds', 'total').replace('turnovers', 'TO')
                for c in feat_cols]

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle mask
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    xticklabels=short_labels,
    yticklabels=short_labels,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS + 'feature_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/feature_correlation_heatmap.png')

---
## Summary Statistics

In [ ]:
feat_cols_only = [c for c in df.columns if c.startswith('home_avg_') or c.startswith('away_avg_')]
df[feat_cols_only + ['target']].describe().round(2)

---
## Key Observations

1. **Home Advantage**: Home teams win ~55–60% of games, reflecting the well-known home field advantage.
2. **Points**: Winning teams consistently average more points scored and allow fewer points than losing teams.
3. **Rushing vs Passing**: Winning home teams tend to have slightly higher rushing yards — balanced offenses correlate with wins.
4. **Correlations**: `home_avg_pts_scored` and `home_avg_total_yards` are positively correlated with `target`. `home_avg_pts_allowed` is negatively correlated.

These observations justify the feature choices for the models in `model.py`.